# Seasonal and static-seascape imputation challengers

This research-only notebook extends the learned marine transport model with richer seasonal variables and a curated static physical seascape. It writes only to `outputs/covariates/`; it cannot update a production model or release pointer.

The evaluation reconstructs the current encounter-held-out folds, excludes each outer test encounter from the anchor pool, tunes regularization inside each outer-training fold, restores natural source/class prevalence through post-stratification, and applies source/era/H3-R4 degradation gates.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / 'covariate_experiment.py').is_file():
    raise RuntimeError('Run this notebook from notebooks/cetaceans/killer_whales/imputation/testing')
import os
REPO_ROOT = Path(os.environ["MARINE_MAMMALS_WORKSPACE_ROOT"]).expanduser().resolve()
for path in (HERE,):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from covariate_experiment import run_covariate_experiment
from experiment_support import resolve_release_paths

paths = resolve_release_paths(REPO_ROOT)
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'covariates'
paths

## Feature contract

Seasonal variables include four annual Fourier harmonics, astronomical daylight length and daily daylight change, circular distance to the winter and summer solstices, and meteorological-season indicators. The interaction arm lets the learned transport score and physical seascape vary smoothly by season.

The physical seascape includes bathymetry, geomorphometry, geomorphic-unit proportions and distances, shoreline/open-water configuration, waterbody morphometry, estuary/fluvial connectivity, and benthic substrate. Current habitat and anthropogenic snapshots are deliberately excluded because they are not time-frozen across the 1980–2026 evaluation period.

In [ ]:
experiment = run_covariate_experiment(output_dir, paths)
comparison = experiment['covariate_comparison'].sort_values(['log_loss', 'brier'])
comparison

## Guardrail results

A candidate is a research recommendation only when every source, era, and H3-R4 region containing at least 100 independent evaluation encounters stays within 10% relative Brier degradation versus the current model. These are stratified diagnostics on the encounter folds, not substitutes for rolling-origin or spatial-blocked outer evaluation.

In [ ]:
experiment['covariate_gate_summary'].sort_values(['model', 'stratification'])

## Seascape coverage

Missing seascape values remain missing through the join, receive fold-local median imputation, and have explicit missingness/cell-availability indicators. They are never converted into observed environmental zeros.

In [ ]:
experiment['covariate_seascape_coverage']

## Selected feature signals

Coefficients are from standardized, L1-regularized fold models. They are useful for stability and ablation inspection, not causal interpretation.

In [ ]:
(experiment['covariate_top_coefficients']
 .groupby(['variant', 'feature'], as_index=False)
 .agg(mean_abs_coefficient=('absolute_coefficient', 'mean'), folds_selected=('outer_fold', 'nunique'))
 .sort_values(['variant', 'mean_abs_coefficient'], ascending=[True, False])
 .groupby('variant', as_index=False)
 .head(15))

In [ ]:
print('Recommended research challenger:', experiment['recommended_research_challenger'])
print('Best feature-augmented challenger:', experiment['recommended_feature_augmented_challenger'])
print('Manifest:', experiment['manifest_path'])
print('Production promotion eligible: False')